# Catalysis case study

We run the full pipeline on the single-atom catalysis dataset (91 metal/oxide
binding energies, 59 primary features) and compare to the published iBART result.

In [ ]:
from jax_ibart import IBart
from jax_ibart.datasets import load_metal_support

X, y, names, units = load_metal_support()
print("X:", X.shape, " y:", y.shape, " features:", len(names))
print("two known drivers present:", "Hfo" in names, "Oxv" in names)

We use the differentiable backend here so the notebook runs in seconds. The
faithful `bart` backend (BART-G.SE) is more accurate but takes minutes; its
results and the published comparison are in
[the validation writeup](../validation-catalysis.md) and
[the oracle cross-check](../oracle-crosscheck.md).

We pass `units=units` to enable the dimensional-analysis filter (it drops
physically meaningless descriptors like `volume + speed`).

In [ ]:
model = IBart(
    backend="diff",
    schedule=("b", "u", "b"),
    max_dim=3,
    n_keep=10,
    seed=0,
    steps=1000,
    l0=0.03,
).fit(X, y, names, units=units)

pred = model.predict(X)
r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(f"in-sample R^2 = {r2:.3f}\n")
for desc, coef in zip(model.descriptors_, model.coef_[1:]):
    print(f"  {coef:+.4g} * {desc}")

### Comparison to the published (reference R-iBART) result

In [ ]:
import json
from pathlib import Path

oracle = json.loads(Path("../../data/oracle_ibart.json").read_text())
print("published canonical descriptor :", oracle["canonical_descriptor"])
print("published l0 1-descriptor model:", oracle["lzero_1"])
print("published l0 3-descriptor model:", oracle["lzero_3"])
print(
    "published l0 RMSEs (k=1..5)    :",
    [round(v, 3) for v in oracle["lzero_in_sample_rmse"]],
)

The reference model's canonical descriptor is `abs(Hfo/Oxv)`, coupling the oxide
formation enthalpy and the oxygen vacancy energy, the metal-oxygen
charge-transfer drivers of metal-support binding. Our port recovers models built
on these same physical drivers; exact-descriptor reproduction needs a permissive
PAN-aligned screen (see the oracle cross-check for the honest details).

## References and data provenance

- **Dataset** (`load_metal_support`): the single-atom catalysis `catalysis` data
  (91 metal/oxide binding energies, 59 features) shipped with the R `iBART`
  package; the underlying DFT data are from the Senftle group's MIT-licensed
  [`Metal-Oxide-LASSO-lo`](https://github.com/tsenftle/Metal-Oxide-LASSO-lo)
  repository. See [`data/README.md`](https://github.com/jkitchin/jax-ibart/blob/main/data/README.md).
- **Reference result** (`oracle_ibart.json`): the published iBART l0 descriptors
  and RMSEs, from the papers below (citable facts; see `NOTICE.md`).
- **Method**: C.-Y. Liu, S. Ye, M. Li, T. P. Senftle, *J. Chem. Phys.* **156**,
  164105 (2022); S. Ye, T. P. Senftle, M. Li, *JASA* **119**:81-94 (2024),
  arXiv:2110.10195.
- Full bibliography: [docs/references.md](../references.md). Honest comparison to
  the reference implementation: [docs/oracle-crosscheck.md](../oracle-crosscheck.md).